In [12]:
import pandas as pd
import re

In [ ]:
D7 = "/Users/user/github-repos/Hackathon_Salzburg/backend/datasets/upperaustria_denkmal_list.csv"

In [14]:
calender_features = pd.read_csv(D1, sep=";")
doi_per_region = pd.read_csv(D2, sep=";")
daily_event_per_region_G = pd.read_csv(D3, sep=";")
daily_event_per_region_M= pd.read_csv(D4, sep=";")
daily_event_per_region_K= pd.read_csv(D5, sep=";")
monthly_regional_scaling= pd.read_csv(D6, sep=";")
denkmal = pd.read_csv(D7, sep=";")
denkmal.head()

,HERIS-ID,Bundesland,Gemeinde,KG,Objekttitel,Adresse,Grundstücksnummern,Denkmalschutz Status
0,18678,Oberösterreich,Adlwang,49001 Adlwang,Pfarrhof,"Bad-Haller-Straße 6 , 4541 Adlwang",.21/1,Denkmalschutz per Verordnung
1,18696,Oberösterreich,Adlwang,49001 Adlwang,"Kath. Filialkirche, Wallfahrtskirche St. Blasien","Grünburger Straße 117 , 4541 Adlwang",.66,Denkmalschutz per Verordnung
2,18676,Oberösterreich,Adlwang,49001 Adlwang,"Mesnerhaus, Blasenhaus","Grünburger Straße 119 , 4541 Adlwang",.65,Denkmalschutz per Verordnung
3,18677,Oberösterreich,Adlwang,49001 Adlwang,"Kath. Pfarrkirche, Wallfahrtskirche Sieben Sch...","Kirchenplatz 1 , 4541 Adlwang",.1,Denkmalschutz per Verordnung
4,18679,Oberösterreich,Adlwang,49001 Adlwang,Pfarrheim,"Kirchenplatz 10 , 4541 Adlwang",.10,Denkmalschutz per Verordnung


In [15]:
# Keyword filter: keep monuments a visitor would actually go and see.
# Categories are checked in order, the first match wins.
CATEGORIES = {
    "museum_memorial": r"(?<!ehem\. )museum|gedenkstätte|memorial|besucherzentrum|konzentrationslager|kz-|geburtshaus|sterbehaus",
    "abbey":           r"\bstift\b|stiftskirche|stiftspfarrkirche|kloster|abtei|basilika|\bdom\b|domkirche",
    "castle":          r"schloss|schloß|schlössl|\bburg\b|burgruine|herzogsburg|\bruine|festung|palais|residenz|\bansitz",
    "archaeology":     r"römer|römisch|lauriacum|legionslager|kastell|vicus|fundzone|gräberfeld|hügelgr|ringwall|pfahlbau|höhensiedlung|burgstall|hallstattkultur|felszeichnung|befunde",
    "pilgrimage":      r"wallfahrt",
    "fortification":   r"stadtmauer|marktmauer|befestigung|stadttor|\btor\b|torturm|torbau|bastion|zwinger|wehrturm|\bturm\b|stadtturm|\bfort\b",
    "landmark":        r"rathaus|pranger|mariensäule|pestsäule|dreifaltigkeitssäule|marktbrunnen|stadtbrunnen|(?<!krieger)denkmal|lichtsäule|totenleuchte",
    "technical":       r"soleleitung|hammer|sensenwerk|pferdeeisenbahn|seilbahn|lokalbahn|pöstlingbergbahn|klause|glockengießerei|salzstadel|salzturm|innerberger stadel|stollen",
    "culture_venue":   r"theater|brucknerhaus|kurhaus|trinkhalle|kaiservilla|musikpavillon|botanischer garten|lehár|stadthalle",
}

# If the main (first) part of the title is an everyday building, drop it,
# even when a later part says e.g. "ehem. Schloss" or "Teil der Stadtmauer".
EXCLUDE_LEAD = (
    r"^(ehem\.\s+|alte[rs]?\s+)?("
    r"bürger|wohn|miet|doppelwohn|arbeiter|kommunal|ackerbürger"
    r"|pfarrhof|pfarrheim|pfarrhaus|pfarrzentrum|stadtpfarrhof|dechanthof|mesner|benefiziat|kaplan"
    r"|.*schule\b|.*gymnasium|kindergarten|(markt)?gemeindeamt|amtsgebäude|bank|sparkasse|nationalbank"
    r"|bezirksgericht|kreisgericht|bezirkshauptmannschaft|bh\s|amt der|landesregierung|.*kammer|kaserne|krankenhaus"
    r"|wirtschaftsgebäude|bauern|meierhof|stiftsmeierhof|.*stiftshaus|stiftshof|freihaus|erdkeller|keller"
    r"|gasthaus|gasthof|hotel|lagergebäude|verwaltung|forsthaus|forsthube|stadel|schüttkasten"
    r"|friedhof|aufbahrungshalle|grenzstein|.*bildstock|figur|kruzifix|kriegerdenkmal|gedenk(tafel|stein)"
    r"|wegkapelle|flur|hofkapelle|nischenkapelle"
    # ordinary parish/branch churches and chapels: one in every village
    r"|kath\.|evang\.|\w*kirche|\w*kapelle|karner|kalvarienberg|\w*kreuzweg"
    r")"
)

# Always keep these, whatever the first part of the title says
# (e.g. "Wohnhaus, Bruckner-Geburtshaus", "Troadkasten, Heimatmuseum",
# "Kath. Pfarrkirche, Wallfahrtskirche ...").
ALWAYS_KEEP = r"(?<!ehem\. )museum|gedenkstätte|memorial|konzentrationslager|kz-|geburtshaus|wallfahrt|basilika|\bdom\b|stiftskirche|stiftspfarrkirche|klosterkirche|klosterbauten"


def classify(title):
    t = str(title).lower()
    lead = t.split(",")[0].strip()
    if not re.search(ALWAYS_KEEP, t) and re.search(EXCLUDE_LEAD, lead):
        return None
    for category, pattern in CATEGORIES.items():
        if re.search(pattern, t):
            return category
    return None


denkmal["category"] = denkmal["Objekttitel"].map(classify)
touristic = denkmal[denkmal["category"].notna()].copy()

In [ ]:
# Clean addresses for geocoding: drop the "#MIG-x#" markers and the

# position hints like "(bei)", "(vor)", "(gegenüber)".
#POSITION_HINT = r"\s*\((bei|vor|gegenüber|hinter|neben|in der nähe|nördlich|südlich|östlich|westlich|nordöstlich|nordwestlich|südöstlich|südwestlich)\)"

#touristic["address_clean"] = (
#    touristic["Adresse"]
#    .str.replace(r"\s*#MIG-\w+#", "", regex=True)
#    .str.replace(POSITION_HINT, "", regex=True, flags=re.IGNORECASE)
#    .str.replace(r"\s+,", ",", regex=True)
#    .str.strip()
#)

#touristic = touristic.dropna(subset=["Adresse"])
print(len(touristic), "of", len(denkmal), "monuments kept")
print(touristic["category"].value_counts())

touristic[["HERIS-ID", "Gemeinde", "Objekttitel", "category", "Adresse"]].to_csv(
    "/Users/user/github-repos/Hackathon_Salzburg/backend/datasets/upperaustria_denkmal_touristic.csv",
    sep=";", index=False,
)

touristic

919 of 5982 monuments kept
category
castle             232
landmark           157
fortification      145
technical           98
museum_memorial     90
archaeology         90
abbey               55
pilgrimage          31
culture_venue       21
Name: count, dtype: int64


,HERIS-ID,Bundesland,Gemeinde,KG,Objekttitel,Adresse,Grundstücksnummern,Denkmalschutz Status,category
1,18696,Oberösterreich,Adlwang,49001 Adlwang,"Kath. Filialkirche, Wallfahrtskirche St. Blasien","Grünburger Straße 117 , 4541 Adlwang",.66,Denkmalschutz per Verordnung,pilgrimage
3,18677,Oberösterreich,Adlwang,49001 Adlwang,"Kath. Pfarrkirche, Wallfahrtskirche Sieben Sch...","Kirchenplatz 1 , 4541 Adlwang",.1,Denkmalschutz per Verordnung,pilgrimage
13,18456,Oberösterreich,Aigen-Schlägl,47001 Aigen,Mariensäule,"Marktplatz 21 , 4160 Aigen-Schlägl (Aigen im M...",2860/1,Denkmalschutz per Verordnung,landmark
19,20149,Oberösterreich,Aigen-Schlägl,47009 Schlägl,"Sommerhaus Stift Schlägl, Musikzentrum St. Nor...","Schaubergstraße 7 , 4160 Aigen-Schlägl (Schlägl)",59,Denkmalschutz per Verordnung,abbey
20,20144,Oberösterreich,Aigen-Schlägl,47009 Schlägl,Prämonstratenserkloster Stift Schlägl,"Schlägl 1 , 4160 Aigen-Schlägl",".20, .21",Denkmalschutz per Verordnung,abbey
...,...,...,...,...,...,...,...,...,...
5933,38912,Oberösterreich,Windischgarsten,49412 Windischgarsten,Römischer Gebäudekomplex,"Kirchfeldstraße 26 , 4580 Windischgarsten (bei)","945, 947/3",Denkmalschutz per Bescheid,archaeology
5936,81168,Oberösterreich,Windischgarsten,49412 Windischgarsten,Heimatmuseum,"Museumstraße 7 , 4580 Windischgarsten",.1,Denkmalschutz per Verordnung,museum_memorial
5945,38268,Oberösterreich,Wolfern,49216 Losensteinleithen,Schloss Losensteinleithen,"Losensteinleiten 1 , 4493 Wolfern (Losensteinl...",10/5,Denkmalschutz per Bescheid (Unterschutzstellun...,castle
5956,38923,Oberösterreich,Wolfsegg am Hausruck,50217 Wolfsegg,Schloss Wolfsegg mit Wirtschaftsgebäude,"Schloßberg 1 , 4902 Wolfsegg am Hausruck",".1, .2/1",Denkmalschutz per Bescheid (Unterschutzstellun...,castle


In [18]:
touristic.to_csv("filtered_touristic.csv", sep=";")